# Audio/Video Splitter
Requires the `media_splitter/` folder next to this notebook and FFmpeg installed.
In Colab, upload the `media_splitter` folder (or a zip of it and unzip) first.

In [5]:
!ffmpeg -version | head -1


'ffmpeg' is not recognized as an internal or external command,
operable program or batch file.


In [ ]:
from pathlib import Path
from media_splitter import SplitConfig, split_media

UPLOAD_DIR = Path("uploads"); UPLOAD_DIR.mkdir(exist_ok=True)

# Upload: Colab dialog if available, otherwise drop files into ./uploads manually.
try:
    from google.colab import files
    for name, data in files.upload().items():
        (UPLOAD_DIR / name).write_bytes(data)
except ImportError:
    print("Not in Colab. Put your files in ./uploads")

inputs = sorted(p for p in UPLOAD_DIR.iterdir() if p.is_file())
print([p.name for p in inputs])

In [ ]:
CHUNK_SECONDS = 30      # change this
AUDIO_FORMAT = "wav"    # wav recommended for mel-spectrograms
ACCURATE_VIDEO = True   # False = faster, but cut points snap to keyframes

cfg = SplitConfig(CHUNK_SECONDS, audio_format=AUDIO_FORMAT, accurate_video=ACCURATE_VIDEO)

def progress(done, total, name):
    print(f"[{done}/{total}] {name}", end="\r")

report = split_media(inputs, cfg, "split_output.zip", on_progress=progress)
print(f"\n{report.total_chunks} chunks -> {report.zip_path}")
for r in report.failed:
    print("FAILED:", r.source, "-", r.error)

In [ ]:
try:
    from google.colab import files
    files.download("split_output.zip")
except ImportError:
    print("Output: split_output.zip")